# darts_forecast day-start: predict a stretch of the trading day from its morning

**In one sentence.** For the explorer's 12 stocks, a Darts model reads the day's bars from a fixed morning
time (09:30 + *O*, for *L* minutes) and predicts the next *H* minutes of the SAME day, bar by bar. The bar size
*g*, the offset *O*, the input *L* and the output *H* are the hyperparameters an optimizer searches (§0), and
every predicted path is graded against "no change".

Every example is one stock-day and stays inside one session: no input and no output ever reaches past the
close or across a night. The model reads the path so far, optionally each bar's **volume**, and optionally two
numbers from the **previous days** (the overnight gap and the previous days' move: examples of daily
information, not chosen by a study).

**Roadmap**

| § | question |
|---|---|
| 0 | which settings, and what do the words mean? |
| 1 | which stocks and which months? |
| 1b | what does one example look like: what goes into the model, what it must predict, what happened? |
| 2 | one configuration end to end: what does the model predict, and does the pipeline do what it says? |
| 3 | the grid: which bar size, offset, input and output predict the path best? |
| 4 | do the optional inputs (volume, the previous days) help? |
| 5 | does the winner hold on a month the grid never saw? |
| 6 | the checks and the summary |

**Run time.** One configuration is one model fit. Every result is saved in `cache/`, so measuring the same
configuration again takes seconds. `MAX_TRIALS` (§0) is the budget of a run.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel, or run  uv sync --all-extras
import importlib.util
import logging
import warnings
from math import prod

warnings.filterwarnings("ignore")
logging.getLogger("darts").setLevel(logging.ERROR)  # Darts' notes about optional models it lacks

import pandas as pd
from IPython.display import Markdown

from trading_simple import strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine


def helpers(name: str):
    "One of the strategy folder's helper modules, loaded by path (strategy.py imports none)."
    path = strategies.STRATEGIES_DIR / "darts_forecast" / f"{name}.py"
    spec = importlib.util.spec_from_file_location(f"darts_{name}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


ds, charts = helpers("daystart"), helpers("charts")
ex, rs = ds.ex, ds.rs  # explorer.py and research.py, as loaded by daystart
settings = load_settings()
engine = make_engine(settings)
pd.set_option("display.max_colwidth", 90)

## 0 · The settings

**Question: which settings, and what do the words mean?**

Every setting is in the next cell, in two groups. Running the cell prints both.

| group | what it holds | who changes it |
|---|---|---|
| **Hyperparameters** (`SPACE`) | the knobs the optimizer chooses: bar size, offset, input, output. Each lists the values it may pick; together they are the **optimization space** | the optimizer, trial by trial |
| **Fixed** | everything else: the model, the stocks, the months, the inputs and the training limits | you, before a run |

A hyperparameter with one value is **fixed but still declared**: the bar size is `[1]` for now. Give it more
values (say `[1, 2, 5, 10]`) and it joins the search.

`MAX_TRIALS` is the budget: the most configurations one run may fit and grade (a **trial** is one configuration,
one model fit). Lower it for a quick test run, raise it for the real one. §3 does not read it yet: until it is
rewritten as the optimizer, §3 still measures every configuration of the space.

In [ ]:
# ── HYPERPARAMETERS: what the optimizer may change ────────────────────────────────────────
# name: ([the values the optimizer may pick], what it is). All in minutes; one value = fixed.
SPACE = {
    "bar": ([1], "bar size: minutes per bar (fixed to 1 for now; list more to search it)"),
    "offset": ([0, 15, 30, 60], "minutes after 09:30 before the input starts"),
    "input": ([15, 30, 60, 120], "minutes the model reads"),
    "output": ([30, 60, 120, 240, 360], "minutes the model predicts"),
}
MAX_TRIALS = 20  # the budget: at most this many configurations are fitted and graded
DEFAULT = (1, 30, 60, 120)  # worked example (§2): (bar, offset, input, output), a point of SPACE

# ── FIXED: the same for every trial ───────────────────────────────────────────────────────
MODEL = "tide"  # the Darts model: tide | tsmixer | dlinear
SYMBOLS = None  # None: the explorer's 12 picks; or your own list, e.g. ["AAPL", "KO", "SMCI"]
WINDOW = ("2025-08-01", "2026-01-31")  # the months studied (inside the darts study's TUNE months)
TRAIN_MONTHS, TEST_MONTHS = 4, 1  # TRAIN: first 4 months; TEST: last month; VALIDATION: the rest
TRAIN_FROM = "2025-06-18"  # the first training day: the first stored 1-minute day (None: TRAIN's)
VOLUME = True  # each bar's volume as an input
PREV = True  # the two previous-day inputs (the overnight gap, the previous days' move)
PREV_DAYS = 1  # the "x" of the previous days' move: over the last x sessions
EPOCHS = 50  # at most this many training passes (stopped early on held-out days)
DRAWS = 200  # make-believe futures behind every luck band

# ── Derived: what the later cells read ────────────────────────────────────────────────────
VALUES = {k: v for k, (v, _) in SPACE.items()}
K = {"window": WINDOW, "train_months": TRAIN_MONTHS, "granularities": VALUES["bar"],
     "offsets": VALUES["offset"], "inputs": VALUES["input"], "outputs": VALUES["output"],
     "epochs": EPOCHS, "draws": DRAWS, "default": DEFAULT, "train_from": TRAIN_FROM}
RUN = {"kind": MODEL, "volume": VOLUME, "prev": PREV, "prev_days": PREV_DAYS,
       "fit_from": K["train_from"], "epochs": K["epochs"], "draws": K["draws"]}
GRID = {"grans": K["granularities"], "offsets": K["offsets"], "inputs": K["inputs"],
        "outputs": K["outputs"]}
CONFIGS = ds.cells(*GRID.values())  # the feasible points of SPACE
if DEFAULT not in CONFIGS:
    raise ValueError(f"DEFAULT {DEFAULT} is not a feasible point of SPACE (whole bars, "
                     f"offset + input + output <= {ds.SESSION} minutes)")
SPAN = {k: f"{a:%Y-%m-%d} → {b:%Y-%m-%d}" for k, (a, b) in ex.months(
    WINDOW, train_months=TRAIN_MONTHS, test_months=TEST_MONTHS).items()}  # the split, by date
CHECKS = []  # every section adds its checks; §6 shows them all

# ── The output: what is searched, what is fixed ───────────────────────────────────────────
display(Markdown("### Hyperparameters: the optimizer's search space"))
display(pd.DataFrame(
    [(k, doc, ", ".join(map(str, v)), len(v), "fixed" if len(v) == 1 else "searched")
     for k, (v, doc) in SPACE.items()],
    columns=["hyperparameter", "what it is", "values (minutes)", "n values", "status"],
).set_index("hyperparameter"))
print(f"space:  {prod(len(v) for v in VALUES.values())} combinations, {len(CONFIGS)} feasible "
      f"(whole bars, offset + input + output <= {ds.SESSION} minutes)")
print(f"budget: MAX_TRIALS = {MAX_TRIALS} (one trial = one configuration fitted and graded)")
print(f"worked example (§2): {ds.label(DEFAULT)}")

display(Markdown("### Fixed: the same for every trial"))
display(pd.DataFrame([
    ("model", MODEL, "the Darts network: tide | tsmixer | dlinear", "this cell"),
    ("stocks", ", ".join(SYMBOLS) if SYMBOLS else "the explorer's 12 picks",
     "the stocks one model learns", "this cell"),
    ("window", f"{WINDOW[0]} → {WINDOW[1]}", "the months studied", "this cell"),
    ("TRAIN", SPAN["train"], "the first fit learns here (and from fit from on, below)",
     "this cell"),
    ("VALIDATION", SPAN["validation"], "the months between: the optimizer is graded here",
     "this cell"),
    ("TEST", SPAN["test"], "the last months: read once, at the end", "this cell"),
    ("fit from", TRAIN_FROM, "the first training day (the first stored 1-minute day)",
     "this cell"),
    ("volume input", VOLUME, "each bar's volume as an input", "this cell"),
    ("previous-day inputs", PREV, "the overnight gap and the previous days' move", "this cell"),
    ("previous days x", PREV_DAYS, "the previous days' move runs over the last x sessions",
     "this cell"),
    ("epochs", EPOCHS, "at most this many training passes (stopped early on held-out days)",
     "this cell"),
    ("luck draws", DRAWS, "make-believe futures behind every luck band", "this cell"),
    ("held-out days", f"{ds.HELD_DAYS['validation']} (TEST refit: {ds.HELD_DAYS['test']})",
     "the last fit days kept out: early stopping and the scale β", "daystart.py"),
    ("typical-size window", f"{ds.VOL_DAYS} sessions",
     "every typical size (moves, volume, gap) reads earlier sessions only", "daystart.py"),
    ("move clip", f"±{ds.CLIP:g} typical moves", "a bigger bar move counts as this", "daystart.py"),
    ("learning rate", ex.SETTINGS["lr"],
     "network size and dropout are fixed in make_path_model too", "explorer.py"),
], columns=["setting", "value", "what it is", "set in"]).set_index("setting").astype(str))

**Words you will meet**

| word | meaning |
|---|---|
| bar, *g* | *g* minutes of trading built from the stored 1-minute bars |
| offset *O* | minutes after the 09:30 open before the input starts (30: the first half hour is skipped) |
| input *L* | the minutes the model reads: the bars from 09:30 + *O* to 09:30 + *O* + *L* |
| output *H* | the minutes it predicts, bar by bar and all at once: the predicted **path** |
| decision | the moment the path is predicted: 09:30 + *O* + *L*, the same time every day for one configuration |
| bar move, typical move | a bar's move inside the day (the day's first bar from its first trade) divided by the usual size of that clock time's move over the last 20 sessions, so calm and wild stocks, the open and midday are comparable. A move beyond 10 typical moves counts as 10, also in the graded truth (it rarely binds) |
| R² against "no change" | 1 − (squared error of the predicted path) / (squared error of predicting no change), per stock and lead: 0 = as good as saying nothing, above 0 = better. **The grade**, averaged over leads and stocks |
| scale β | how much of the network's path survives held-out days: 0 = none (the path becomes "no change"), 1 = all |
| direction (corr), z | the correlation of predicted and actual change (the scale cannot move it), and how many luck-SDs it sits above luck |

**What the model reads** (all known at the decision):

- **The path so far:** the input bars' moves, in typical moves. Within one configuration every example starts at
  the same clock time, so a bar's place in the window IS its time of day: no calendar input is needed.
- **Volume** (`VOLUME`): each input bar's volume against the stock's average bar over the last 20 sessions (log).
  The busy open and close stay in it.
- **The previous days** (`PREV`): two numbers for the whole day, examples of daily information, not chosen by a
  study:
  - **gap**: today's first trade against yesterday's last, in typical nights (one earnings night cannot shrink
    the next twenty). It is withheld when the stock had not traded before the decision.
  - **previous days' move**: the last `PREV_DAYS` sessions' close-to-close move, in typical moves.
- **Which stock** (one-hot): one model learns all the stocks.

**What can go wrong**, and the guard:

| trap | guard |
|---|---|
| a stale opening print (a volume-0 placeholder: nobody traded) fakes the first bar and the gap | volume-0 minutes count as no trade |
| a network that learns noise draws paths too loud for what it knows | training stops early on held-out days, and its paths are scaled by β from them |
| the best of 372 tries flatters itself | a luck band for direction, the bar for the best of all the tries, and a TEST month read once |
| a typical size computed with later data leaks the future | every size uses earlier sessions only; the leak check of §2 can fail |
| 1-minute bars bounce between bid and ask | the AR(1) baseline is graded beside every configuration; do not over-read the bar-size ranking |

## 1 · The stocks, the months and the split

**Question: which stocks and which months?**

The explorer's 12 picks (two per market-cap group and speed, complete 1-minute data; `darts_forecast_explorer.ipynb`
§1 explains the rules) and its window and split: **TRAIN** (the fit learns here), **VALIDATION** (graded; the
grid chooses here) and **TEST** (read once, §5). With `TRAIN_FROM` the fit also learns from the days before TRAIN,
back to the first stored 1-minute day (2025-06-18): the table below calls them the warm-up. A day trains once the
typical sizes have 10 earlier sessions (from 2025-07-07 here), so the first stored days serve only as history.
Half-days (13:00 close) are never used.

In [ ]:
cands = ex.candidates(engine, WINDOW)  # always the full window: its saved scans exist
picks = ex.pick(cands)
NAMES = SYMBOLS or list(picks["symbol"])
panel, SP = ex.load_window(settings, engine, NAMES, K["window"], train_months=K["train_months"],
                           test_months=TEST_MONTHS,
                           warmup=ds.warmup(K["train_from"], K["window"][0]))
fit_d, held_d, _ = ds.split_days(ds.prepared(panel), SP, "validation", K["train_from"])
dates = ds.prepared(panel)["dates"]
print(f"{len(panel.symbols)} stocks, {len(SP['sessions'])} sessions {SP['sessions'][0]} → "
      f"{SP['sessions'][-1]}; the fit learns from {dates[fit_d[0]]} → {dates[fit_d[-1]]} "
      f"({len(fit_d)} days), held out {dates[held_d[0]]} → {dates[held_d[-1]]} ({len(held_d)})")
display(ex.pick_table(picks[picks["symbol"].isin(NAMES)]))
display(ex.split_table(SP))

## 1b · What goes in, what comes out

**Question: what does one example look like: what the model reads, what it must predict, and what actually happened?**

One example is one stock-day. The figure draws the worked example (`DEFAULT`) for one stock and one day, **before
any training**:

- **Top: the day's price.** Grey at the start: the **offset**, skipped. Blue: the **input** window, the bars the
  model reads. Green: the **output** window, the bars it must predict. Light grey: the rest of the day, not used.
- **Middle: the target, bar by bar,** in typical moves (see "Words you will meet" in §0). Blue bars are the input,
  green bars are the **truth**: what the stock actually did in the output window. The dashed line is the slot for
  the **model's output**: until a model is trained it shows "no change" (0), the baseline every model is graded
  against.
- **Bottom: the volume input** (`VOLUME`), for the input bars only. The model never sees the output window's volume.

The static inputs are single numbers for the whole day (the stock, and with `PREV` the gap and the previous days'
move). They are printed with the model that will read all of this. Pick another stock or day with `PICK_STOCK` and
`PICK_DAY`.

In [ ]:
PICK_STOCK = None  # None: a fast large cap; or any symbol of NAMES
PICK_DAY = None  # None: the 5th VALIDATION session; or any session, "YYYY-MM-DD"

g0, o0, l0, h0 = K["default"]
pp = ds.prepared(panel)
name0, ts0 = ex.example(picks, NAMES, SP, clock=ds.clock(o0 + l0))
NAME_IO, DAY_IO = PICK_STOCK or name0, PICK_DAY or ts0[:10]
d_io, k_io = list(pp["dates"]).index(DAY_IO), pp["names"].index(NAME_IO)
# exactly what Darts is given for this stock-day: the input bars followed by the actual output bars
ser, vol, _, _ = ds.examples(ds.bars(pp, g0), pp, K["default"], [d_io], full=True, volume=VOLUME,
                             prev=PREV, prev_days=PREV_DAYS)
y, static = ser[k_io].values()[:, 0], ser[k_io].static_covariates.iloc[0]
IO = {"name": NAME_IO, "day": DAY_IO, "cell": K["default"],
      "times": pd.date_range(f"{DAY_IO} 09:31", periods=ds.SESSION, freq="min"),
      "price": pp["cf"][d_io, :, k_io], "y_in": y[: l0 // g0], "y_out": y[l0 // g0:],
      "volume": vol[k_io].values()[: l0 // g0, 0] if VOLUME else None, "pred": None}
print(f"{NAME_IO} on {DAY_IO}: {ds.label(K['default'])}")
charts.model_io(IO).show()

In [ ]:
est = ex.make_path_model(MODEL, l0 // g0, h0 // g0, epochs=EPOCHS,
                         n_samples=len(fit_d) * len(NAMES))  # as ds.fit builds it; nothing trained
if (est.input_chunk_length, est.output_chunk_length) != (len(IO["y_in"]), len(IO["y_out"])):
    raise ValueError("the model's input / output length differs from the example's")
static_what = f"the stock (one-hot: is_{NAME_IO} = 1)" + (
    f", the gap ({static['gap']:+.2f} typical nights) and the previous days' move "
    f"({static['prev_move']:+.2f} typical moves)" if PREV else "")

display(Markdown(f"### The model: `{type(est).__name__}`, defined and not trained"))
display(pd.DataFrame([
    ("target (past)", "the input bars' moves, in typical moves",
     f"{est.input_chunk_length} bars × 1"),
    ("volume (past covariate)", "the input bars' volume, log against the usual bar",
     f"{len(IO['volume'])} bars × 1" if VOLUME else "off"),
    ("static", static_what, f"{len(static)} numbers"),
    ("output", "the next bar moves, all at once", f"{est.output_chunk_length} bars × 1"),
    ("truth", "the bars the stock actually made: the grade compares them with the output",
     f"{len(IO['y_out'])} bars × 1"),
], columns=["role", "what", "size"]).set_index("role"))
ARCH = ("hidden_size", "num_encoder_layers", "num_decoder_layers", "decoder_output_dim",
        "temporal_decoder_hidden", "ff_size", "num_blocks", "kernel_size", "dropout", "batch_size",
        "n_epochs")  # whichever of these the chosen model has
display(pd.Series({k: est.model_params[k] for k in ARCH if k in est.model_params},
                  name="value").to_frame())

## 2 · One configuration end to end

**Question: for one configuration, what does the model predict, and does the pipeline do what it says?**

The worked example (`DEFAULT`): 1-minute bars, skip 09:30–10:00, read 10:00–11:00, predict 11:00–13:00. One model
is fit for all the stocks on every day from `TRAIN_FROM` to the end of TRAIN (its last 10 days held out: they
stop the training early and set the scale β), then every VALIDATION stock-day gets a predicted path.

**The checks.** The boundaries as built (the fit days inside TRAIN and before the held-out days, those before
VALIDATION; every example inside one regular session), and the **leak check**: every 1-minute bar from the
decision of a VALIDATION day about two weeks in (the check prints the date) on is replaced by random bars and the whole configuration re-run (inputs, sizes, fit, β,
predictions). Every path predicted up to that decision must be identical, and so must β; the later paths must
change (else the check could not fail).

In [ ]:
r0 = ds.run(panel, SP, K["default"], **RUN)
CHECKS += r0["checks"]
CHECKS += ds.truncation(panel, SP, K["default"], **RUN)
display(rs.checks_table(CHECKS))
display(pd.Series(ds.row(r0)).drop(["r2_h", "corr_h"]).to_frame("worked example").T)
print(r0["fit"])

**The figure.** One stock on one VALIDATION day: the actual price (1-minute closes, black) from the open, and the
model's predicted price path from the decision (solid: the graded, scaled path; dotted: the network's own, before
scaling), converted to dollars with each bar's typical size as known at the decision.

In [ ]:
g0, o0, l0, h0 = K["default"]
NAME, TS = ex.example(picks, NAMES, SP, clock=ds.clock(o0 + l0))  # a fast large cap, VALIDATION
charts.path_picture(panel, ds.example_paths(panel, [(ds.label(K["default"]), r0)], NAME, TS[:10]),
                    NAME, TS, before=o0 + l0).show()

**The grade.** For every stock and every lead (1 … *H*/*g* bars after the decision) the predicted path is compared
with the actual one over all the graded days: **R² = 1 − Σ (predicted − actual)² / Σ actual²**, so predicting "no
change" scores exactly 0. The grade is the mean over the leads, then over the stocks (equal weights).
**Direction** is the correlation of predicted and actual change; its **luck band** grades the same paths against
200 make-believe futures in which every bar keeps its size but gets a random sign (one coin per bar and day, shared
by the stocks). **z** is how many luck-SDs the direction sits above luck's centre. The figure's legend counts
BARS, as in the explorer: "1 min (input 60, predict 120)" is 1-minute bars, 60 minutes read and 120 predicted.

In [ ]:
charts.lead_curves(pd.DataFrame([ds.row(r0)]),
                   title="The worked example: R² by lead time (VALIDATION)").show()

## 3 · The grid: bar size × offset × input × output

**Question: which configuration's predicted paths agree best with the actual ones on VALIDATION?**

| knob | steps (minutes) |
|---|---|
| bar size *g* | 1 · 2 · 3 · 5 · 10 · 15 · 30 |
| offset *O* | 0 · 15 · 30 · 60 |
| input *L* | 15 · 30 · 60 · 120 |
| output *H* | 30 · 60 · 120 · 240 · 360 |

A configuration needs whole bars and *O* + *L* + *H* ≤ 390 minutes (one session): 372 of them, each measured once
(a fit takes seconds, so no optimizer is needed). **Read with care:** every (*O*, *L*) asks about a different
stretch of the day, and with 372 tries a few look good by luck. The best direction z must clear the bar for the
best of all the tries (about 3.8), and the winner is read once on TEST (§5). Ties at R² = 0 (scaled to "no
change") are ranked by direction z.

In [ ]:
res = ds.search(panel, SP, **GRID, **RUN)
display(ds.results_table(res).head(15))
print(ds.words(res))

**Each knob on its own.** One panel per knob: every configuration (grey dots), the best (solid) and the average
(dotted) of each step. First the grade (R²; 0 = "no change"), then the direction's z against luck (grey = where
one configuration lands 95% of the time by luck, dashed = the bar the best of a step's tries must clear).

In [ ]:
PARAMS = {"granularity": ("bar size (min)", K["granularities"]),
          "offset_min": ("offset after 09:30 (min)", K["offsets"]),
          "input_min": ("input (min)", K["inputs"]), "output_min": ("output (min)", K["outputs"])}
charts.param_effects(res, PARAMS).show()
charts.param_effects(res, PARAMS, metric="z").show()

**The maps, at the winner's offset.** One small map per bar size: input up, output across, coloured by R² on one
shared scale (blue above 0, red below), each cell with its R² and direction z; the winner is framed.

In [ ]:
WIN = tuple(int(ex.best(res).iloc[0][c]) for c in ("granularity", "offset_min", "input_min",
                                                    "output_min"))
at = res[res["offset_min"] == WIN[1]].assign(input=lambda x: x["input_min"],
                                              prediction=lambda x: x["output_min"])
print(f"winner: {ds.label(WIN)} (offset {WIN[1]} minutes)")
charts.granularity_maps(at, K["granularities"], K["inputs"], K["outputs"], unit="min").show()

**Agreement by lead time.** Every configuration (grey) and the best of each bar size (colour): R² against "no
change" at every point of the path, against how many minutes after the decision it is (log scale). The legend
counts bars (as in §2); each curve's offset and minutes are in the grid's table.

In [ ]:
charts.lead_curves(res, title="R² by lead time: the best configuration of each bar size "
                   "(VALIDATION)").show()

## 4 · Do the optional inputs help?

**Question: does the model use the volume and the previous days' information?**

The worked example and the winner are re-run with each optional input switched off in turn, everything else
unchanged. **Read the direction** (Δcorr, Δz): the scale β often turns R² into exactly 0 either way. A change
inside luck (|Δz| below about 2) is not evidence, and these rows are read after the winner was chosen on the same
months. The tests (`tests/test_darts_forecast_helpers.py`) plant a gap effect and a volume effect in synthetic data
and check that the model picks up both, so "no help" here is about these stocks and months, not the plumbing.

In [ ]:
abl = ds.ablation(panel, SP, [("worked example", K["default"]), ("winner", WIN)], **RUN)
display(abl[["config", "inputs", "r2", "beta", "corr", "z", "Δr2", "Δcorr", "Δz"]].round(4))

## 5 · TEST: does the winner hold on a month the grid never saw?

TEST is graded **once** here, for the winner as set and with each optional input off, each refit on every day
from `TRAIN_FROM` through VALIDATION (the last 20 days before TEST held out for early stopping and β: late December is holiday-thin). The
winner **holds** if its TEST R² is above 0 and its direction z is at least 1.65 (a 5% chance by luck). No TEST
price, path or grade has been shown before this cell.

In [ ]:
tc = ds.ablation(panel, SP, [("winner", WIN)], part="test", **RUN)
display(tc[["config", "inputs", "r2", "beta", "corr", "z", "Δr2", "Δcorr", "Δz"]].round(4))
held = pd.DataFrame([{"config": f"winner, {ds.label(WIN)}", "part": p,
                      **ds.row(ds.run(panel, SP, WIN, part=p, **RUN))}
                     for p in ("validation", "test")])
print(ex.test_words(held.assign(input=held["input_min"], prediction=held["output_min"])))
charts.dots(ex.as_dots(tc, [f"TEST · {i}" for i in tc["inputs"]]), "corr", scale=1.0, unit="",
            title="Direction against luck on TEST: the winner, and without each input").show()

## 6 · The checks and the summary

In [ ]:
table = rs.checks_table(CHECKS)
display(table)
win, test = ex.best(res).iloc[0], held[held["part"] == "test"].iloc[0]
rs.scorecard({"notebook": "daystart", "model": MODEL, "names": NAMES,
              "window": SP["window"], "cells": len(res), "volume": VOLUME, "prev": PREV,
              "prev_days": PREV_DAYS,
              "winner": {"cell": WIN, "val_r2": win["r2"], "val_z": win["z"],
                         "beta": win["beta"], "test_r2": test["r2"], "test_z": test["z"]},
              "above_no_change": int((res["r2"] > 0).sum()),
              "direction_above_luck": int((res["z"] >= 2).sum()),
              "checks_ok": f"{int(table['ok'].sum())}/{len(table)}"})

**Reading the result.** Three outcomes are possible:

- **A configuration beats "no change" and holds on TEST:** that (bar size, offset, input, output) is the baseline
  to build on.
- **A winner that does not hold:** the usual face of luck after many tries.
- **Nothing beats "no change":** for these stocks and months, the next stretch of the day is best predicted as "no
  change", whatever the bar size, offset and lengths. That is itself the baseline.

**What to try next, one knob at a time:**

| next knob | how | why |
|---|---|---|
| more stocks | `SYMBOLS` (about 270 pass the explorer's data rules) | every luck band shrinks, and the model sees more days |
| the model | `MODEL` | TSMixer and DLinear read short inputs differently from TiDE |
| other daily inputs | `daystart.previous` | the two inputs here are examples; any number known at 09:30 fits the same slot |
| more months | the 1-minute data begins 2025-06-18 (`TRAIN_FROM` already starts there); later months are the locked TEST / HOLDOUT | one example per stock-day: months are the scarce resource |